# Swarm-MeZO — Day 4: reputational consensus on RoBERTa+SST-2 (Colab)

Запускает `scripts/run_reputation.py` на удалённой GPU в Colab.

**Что делает прогон:** N=8 агентов, non-IID Dirichlet(α=0.5) split, K=100 локальных MeZO-шагов между раундами консенсуса, 5000 шагов всего; sweep по `β ∈ {0, 0.1, 0.5, 1, 10}` с репутационной матрицей `W_ij = r_j/Σr`.

**Время:** ≈1.5ч на конфиг в bf16 на L4/A100, ≈3ч на T4 (нет bf16 tensor cores → fp32 fallback). Итого 5 конфигов = 7–15 часов.

**Перед запуском:** Runtime → Change runtime type → GPU (желательно A100/L4/V100, T4 пойдёт но медленно).

## 1. Setup: клонирование и зависимости

In [ ]:
!nvidia-smi | head -20

In [ ]:
# Клонируем репозиторий (публичный, доступ без токена).
import os, subprocess
REPO_DIR = '/content/swarm-mezo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/olegkeatzin/swarm-mezo.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull
%cd $REPO_DIR

In [ ]:
# Зависимости — на Colab используем pip, а не uv (уже есть torch с правильной CUDA).
# Если по какой-то причине нужна свежая версия torch — раскомментировать строку ниже.
# !pip install -q --upgrade torch
!pip install -q 'transformers>=4.40' 'datasets>=2.20' tqdm matplotlib pandas
import torch
print(f'torch {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu: {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability(0)
    print(f'compute capability: {cap}  (bf16 fast on cap >= (8,0))')

## 2. Прогрев кеша: модель + датасет

Скачивает RoBERTa-base и SST-2 в `~/.cache/huggingface` чтобы во время прогона не было сетевых задержек.

In [ ]:
from datasets import load_dataset            # ВАЖНО: импортируется ДО torch в скриптах
from transformers import AutoModelForMaskedLM, AutoTokenizer

_ = load_dataset('glue', 'sst2')
_ = AutoTokenizer.from_pretrained('roberta-base')
_ = AutoModelForMaskedLM.from_pretrained('roberta-base')
print('cache OK')

## 3. Санитарный smoke-test (≈1 минута)

10 шагов × N=2 чтобы убедиться что vmap + bf16 + reputation-путь не падают на этой GPU.

In [ ]:
!python scripts/smoke_test_reputation.py 2>&1 | tail -40

## 4. Главный прогон

Скрипт идемпотентный — пишет результаты инкрементально в `outputs/day5_reputation.json` после каждого `β`. Если Colab разорвёт сессию посередине, перезапуск этой ячейки продолжит с недосчитанного.

Запускаем через `tee`, чтобы был и лог на диске, и output в Colab UI.

In [ ]:
!mkdir -p outputs && python -u scripts/run_reputation.py 2>&1 | tee -a outputs/day5_reputation.log

## 5. Быстрый превью результатов

Финальная val accuracy и эволюция репутаций для каждого `β` — диагностика каскада.

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

d = json.loads(open('outputs/day5_reputation.json').read())
runs = dict(sorted(d['runs'].items(), key=lambda kv: kv[1]['beta']))
print(f'{len(runs)} configs in {list(runs.keys())}')
print()
print(f"{'β':>6} | {'final val_acc':>14} | {'final loss':>11} | {'max rep share':>14}")
for name, h in runs.items():
    reps = h.get('reputations') or []
    if reps:
        last = np.asarray(reps[-1])
        max_share = float(last.max() / last.sum())
    else:
        max_share = float('nan')
    print(f"{h['beta']:>6} | {h['eval_acc'][-1]:>14.4f} | {h['eval_loss'][-1]:>11.4f} | {max_share:>14.3f}")

In [ ]:
# Кривые val accuracy на лету (полный анализ — в notebooks/05_day4_reputation.ipynb)
fig, ax = plt.subplots(figsize=(8, 5), dpi=110)
cmap = plt.get_cmap('viridis')
for i, (name, h) in enumerate(runs.items()):
    color = cmap(i / max(len(runs) - 1, 1))
    ax.plot(h['eval_step'], h['eval_acc'], marker='o', markersize=3, color=color,
            label=f"β = {h['beta']}")
ax.set_xlabel('per-agent step')
ax.set_ylabel('SST-2 val accuracy')
ax.set_title('Day 4: reputational MeZO on RoBERTa+SST-2')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

## 6. Скачивание результатов

Заберите JSON + лог себе локально, чтобы прогнать полную визуализацию в `notebooks/05_day4_reputation.ipynb`.

In [ ]:
from google.colab import files
files.download('outputs/day5_reputation.json')
files.download('outputs/day5_reputation.log')

**Альтернатива** — закоммитить и запушить результаты прямо из Colab (потребуется GitHub PAT в `userdata`):

```python
from google.colab import userdata
token = userdata.get('GITHUB_PAT')
!cd /content/swarm-mezo && \
  git config user.email 'colab@local' && \
  git config user.name 'Colab Runner' && \
  git add outputs/day5_reputation.json outputs/day5_reputation.log && \
  git commit -m 'Day 4: reputational β sweep on RoBERTa+SST-2' && \
  git push https://$token@github.com/olegkeatzin/swarm-mezo.git main
```